# LeetCode 242 — Valid Anagram bằng Python

- Link đề: https://leetcode.com/problems/valid-anagram/
- Độ khó: **Easy** · Chủ đề: Hash Table, String, Sorting

Chữ ký hàm LeetCode:

```python
class Solution:
    def isAnagram(self, s: str, t: str) -> bool:

```

**Mục tiêu học**

1. Hiểu định nghĩa anagram: cùng **bộ ký tự** với cùng **số lần xuất hiện**, chỉ khác thứ tự.
2. Ôn `sorted`, `len`, `dict`, `dict.get`, `collections.Counter`, `ord`/`chr`.
3. Đi từ cách sắp xếp (`O(n log n)`) tới **đếm tần suất** (`O(n)`).
4. Biết tối ưu bằng mảng 26 phần tử và một bảng đếm duy nhất (tăng/giảm).
5. Trả lời Follow-up: chuỗi Unicode thì làm sao?

## 1. Đề bài và ví dụ

Cho hai chuỗi `s` và `t`. Trả `True` nếu `t` là **anagram** của `s` (sắp xếp lại các chữ cái của `s` thì được `t`), ngược lại `False`.

| `s` | `t` | Kết quả | Lý do |
|---|---|---|---|
| `"anagram"` | `"nagaram"` | `True` | cùng: a×3, n×1, g×1, r×1, m×1 |
| `"rat"` | `"car"` | `False` | `s` có `t`, `t` có `c` |
| `"aab"` | `"abb"` | `False` | cùng bộ chữ {a,b} nhưng **số lần** khác |
| `"ab"` | `"abc"` | `False` | khác độ dài |

**Constraints:** `1 <= s.length, t.length <= 5 * 10^4`; `s` và `t` chỉ gồm chữ thường tiếng Anh.

**Follow up:** nếu input chứa ký tự Unicode thì điều chỉnh thế nào?

In [1]:
s, t = "anagram", "nagaram"
print(len(s), len(t))
print(sorted(s))
print(sorted(t))

7 7
['a', 'a', 'a', 'g', 'm', 'n', 'r']
['a', 'a', 'a', 'g', 'm', 'n', 'r']


## 2. Nền tảng Python

### 2.1. `sorted(chuỗi)` trả về list ký tự đã sắp xếp

`sorted` nhận bất kỳ iterable nào (chuỗi, list, ...) và trả **list mới** đã sắp xếp tăng dần. Chuỗi được sắp theo mã ký tự (`'a' < 'b' < ...`).

In [2]:
print(sorted("rat"), sorted("tar"), sorted("rat") == sorted("tar"))
print("".join(sorted("nagaram")))   # nối lại thành chuỗi

['a', 'r', 't'] ['a', 'r', 't'] True
aaagmnr


### 2.2. `dict` để đếm tần suất

Ý tưởng "đếm": key là ký tự, value là số lần xuất hiện. `d.get(key, 0)` trả `d[key]` nếu có, ngược lại trả `0` (không gây `KeyError`).

In [3]:
count = {}
for ch in "banana":
    count[ch] = count.get(ch, 0) + 1
    print(f"đọc {ch!r} -> {count}")

đọc 'b' -> {'b': 1}
đọc 'a' -> {'b': 1, 'a': 1}
đọc 'n' -> {'b': 1, 'a': 1, 'n': 1}
đọc 'a' -> {'b': 1, 'a': 2, 'n': 1}
đọc 'n' -> {'b': 1, 'a': 2, 'n': 2}
đọc 'a' -> {'b': 1, 'a': 3, 'n': 2}


`{ch!r}` trong f-string in giá trị kèm dấu nháy (`repr`), giúp nhìn rõ đó là chuỗi.

### 2.3. `collections.Counter` — máy đếm có sẵn

`Counter(iterable)` trả một dict đặc biệt đã đếm sẵn. Hai Counter so sánh bằng `==` khi mọi key có cùng số đếm.

In [4]:
from collections import Counter
c1 = Counter("anagram")
c2 = Counter("nagaram")
print(c1)
print(c1 == c2, Counter("aab") == Counter("abb"))
print("số chữ 'a':", c1["a"], "| chữ không có 'z':", c1["z"])   # key thiếu -> 0

Counter({'a': 3, 'n': 1, 'g': 1, 'r': 1, 'm': 1})
True False
số chữ 'a': 3 | chữ không có 'z': 0


### 2.4. `ord` và `chr` — đổi ký tự ↔ số

`ord('a') = 97`, `ord('b') = 98`, ... Vậy `ord(ch) - ord('a')` cho vị trí `0..25` của chữ thường — dùng làm **index mảng**.

In [5]:
for ch in "abcz":
    print(ch, ord(ch), "-> index", ord(ch) - ord("a"))
print(chr(97 + 2))   # 'c'

a 97 -> index 0
b 98 -> index 1
c 99 -> index 2
z 122 -> index 25
c


## 3. Cách 1 — Sắp xếp rồi so sánh (baseline)

Nếu hai chuỗi là anagram, khi sắp xếp chúng phải thành **cùng một chuỗi** (vì sắp xếp xóa mọi khác biệt về thứ tự). Ngược lại, nếu sắp xếp ra giống nhau thì chúng có cùng ký tự cùng số lần.

In [6]:
def is_anagram_sort(s: str, t: str) -> bool:
    return sorted(s) == sorted(t)

for a, b in [("anagram", "nagaram"), ("rat", "car"), ("aab", "abb"), ("ab", "abc")]:
    print(f"{a!r:>10} vs {b!r:<10} -> {is_anagram_sort(a, b)}")

 'anagram' vs 'nagaram'  -> True
     'rat' vs 'car'      -> False
     'aab' vs 'abb'      -> False
      'ab' vs 'abc'      -> False


### Nút thắt

- Sắp xếp tốn `O(n log n)` và tạo hai list mới (`O(n)` bộ nhớ).
- Ta không thực sự cần **thứ tự**; ta chỉ cần biết **mỗi chữ xuất hiện mấy lần**. Đếm thì làm được trong một lần duyệt `O(n)`.

In [7]:
import timeit
big_s = "abcdefghijklmnopqrstuvwxyz" * 2000
big_t = big_s[::-1]
print("sort   :", round(timeit.timeit(lambda: sorted(big_s) == sorted(big_t), number=50), 4), "giây / 50 lần")
print("Counter:", round(timeit.timeit(lambda: Counter(big_s) == Counter(big_t), number=50), 4), "giây / 50 lần")

sort   : 0.3998 giây / 50 lần
Counter: 0.148 giây / 50 lần


(Thời gian cụ thể tùy máy; với chuỗi rất dài, cách đếm thắng rõ hơn vì `O(n)` so với `O(n log n)`.)

## 4. Insight — anagram ⇔ cùng bảng tần suất

Phát biểu lại đề: `t` là anagram của `s` **khi và chỉ khi** với **mọi** ký tự `c`, số lần `c` xuất hiện trong `s` bằng số lần trong `t`.

Bảng tần suất cho ví dụ:

| ký tự | a | g | m | n | r |
|---|---|---|---|---|---|
| `anagram` | 3 | 1 | 1 | 1 | 1 |
| `nagaram` | 3 | 1 | 1 | 1 | 1 |

Mẹo kiểm tra sớm: nếu `len(s) != len(t)` thì chắc chắn `False` — một dòng giúp thoát nhanh.

## 5. Cách 2 — Hai dict đếm

In [8]:
def build_count(text: str) -> dict:
    count = {}
    for ch in text:
        count[ch] = count.get(ch, 0) + 1
    return count

def is_anagram_two_dicts(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False
    return build_count(s) == build_count(t)

print(build_count("anagram"))
print(is_anagram_two_dicts("anagram", "nagaram"), is_anagram_two_dicts("rat", "car"))

{'a': 3, 'n': 1, 'g': 1, 'r': 1, 'm': 1}
True False


So sánh `dict1 == dict2` trong Python kiểm tra cùng tập key và cùng value cho mỗi key, **không quan tâm thứ tự thêm key**.

In [9]:
print({"a": 1, "b": 2} == {"b": 2, "a": 1})

True


## 6. Lời giải chính — Một bảng đếm, `+1` cho `s`, `-1` cho `t`

Thay vì hai bảng, dùng **một** mảng `count` 26 ô:

1. Nếu độ dài khác → `False`.
2. Duyệt `i` từ `0` tới `n-1`: `count[s[i]] += 1`, `count[t[i]] -= 1`.
3. Anagram ⇔ cuối cùng **mọi ô bằng 0** (mỗi chữ "vào" bao nhiêu lần thì "ra" bấy nhiêu lần).

Dùng mảng (list 26 số) thay dict vì đề chỉ có chữ thường: truy cập index nhanh và bộ nhớ cố định.

In [10]:
def is_anagram_array(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False
    count = [0] * 26                      # list 26 số 0
    base = ord("a")
    for a, b in zip(s, t):
        count[ord(a) - base] += 1
        count[ord(b) - base] -= 1
    return all(c == 0 for c in count)

for a, b in [("anagram", "nagaram"), ("rat", "car"), ("aab", "abb"), ("a", "a")]:
    print(a, b, "->", is_anagram_array(a, b))

anagram nagaram -> True
rat car -> False
aab abb -> False
a a -> True


Cú pháp mới:

- `[0] * 26` tạo list 26 phần tử bằng 0.
- `zip(s, t)` duyệt song song từng cặp ký tự cùng vị trí (vì đã chắc `len` bằng nhau).
- `all(điều_kiện for ...)` trả `True` nếu **mọi** phần tử thỏa (và dừng sớm khi gặp phần tử sai).

In [11]:
print([0] * 5)
print(list(zip("abc", "xyz")))
print(all(x == 0 for x in [0, 0, 0]), all(x == 0 for x in [0, 1, 0]))

[0, 0, 0, 0, 0]
[('a', 'x'), ('b', 'y'), ('c', 'z')]
True False


## 7. Trace từng bước

Trace `s = "aab"`, `t = "abb"` (chỉ hiện ô a, b):

| i | `s[i]` | `t[i]` | count[a] | count[b] | Ghi chú |
|---|---|---|---|---|---|
| 0 | a | a | 0 | 0 | +1 rồi -1 cùng ô a |
| 1 | a | b | 1 | -1 | a dư, b thiếu |
| 2 | b | b | 1 | -1 | +1 rồi -1 ô b |
| cuối | | | **1** | **-1** | có ô ≠ 0 → `False` |

In [12]:
def trace_anagram(s: str, t: str) -> bool:
    if len(s) != len(t):
        print(f"len khác nhau ({len(s)} vs {len(t)}) -> False")
        return False
    count = [0] * 26
    for i, (a, b) in enumerate(zip(s, t)):
        count[ord(a) - 97] += 1
        count[ord(b) - 97] -= 1
        nonzero = {chr(97 + k): v for k, v in enumerate(count) if v != 0}
        print(f"i={i}: +{a} -{b} -> các ô khác 0: {nonzero}")
    ok = all(c == 0 for c in count)
    print("Kết quả:", ok)
    return ok

trace_anagram("aab", "abb")
print()
trace_anagram("anagram", "nagaram")

i=0: +a -a -> các ô khác 0: {}
i=1: +a -b -> các ô khác 0: {'a': 1, 'b': -1}
i=2: +b -b -> các ô khác 0: {'a': 1, 'b': -1}
Kết quả: False

i=0: +a -n -> các ô khác 0: {'a': 1, 'n': -1}
i=1: +n -a -> các ô khác 0: {}
i=2: +a -g -> các ô khác 0: {'a': 1, 'g': -1}
i=3: +g -a -> các ô khác 0: {}
i=4: +r -r -> các ô khác 0: {}
i=5: +a -a -> các ô khác 0: {}
i=6: +m -m -> các ô khác 0: {}
Kết quả: True


True

`{chr(97 + k): v for k, v in enumerate(count) if v != 0}` là **dict comprehension**: tạo dict chỉ chứa các ô khác 0 để in gọn. `enumerate(zip(s, t))` cho `(i, (a, b))` — giải nén lồng bằng `i, (a, b)`.

Ở `"anagram"` các ô dao động nhưng về 0 hết ở cuối — điều quan trọng là **trạng thái cuối**, không phải trạng thái giữa chừng.

## 8. Cách 4 — `Counter` một dòng và cách "trừ dần" có dừng sớm

In [13]:
def is_anagram_counter(s: str, t: str) -> bool:
    return Counter(s) == Counter(t)

def is_anagram_early_exit(s: str, t: str) -> bool:
    if len(s) != len(t):
        return False
    count = Counter(s)
    for ch in t:
        if count[ch] == 0:          # t dùng ch nhiều hơn s có -> dừng ngay
            return False
        count[ch] -= 1
    return True                     # len bằng nhau + không ô nào âm => mọi ô = 0

print(is_anagram_counter("anagram", "nagaram"), is_anagram_early_exit("rat", "car"))

True False


Vì sao `is_anagram_early_exit` không cần kiểm tra "mọi ô = 0" ở cuối? Tổng các ô ban đầu là `len(s)`; mỗi bước trừ 1, sau `len(t) = len(s)` bước tổng còn `0`. Không ô nào âm (đã chặn) mà tổng bằng 0 ⇒ mọi ô bằng 0.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def isAnagram(self, s: str, t: str) -> bool:
        if len(s) != len(t):
            return False
        count = [0] * 26
        for a, b in zip(s, t):
            count[ord(a) - 97] += 1
            count[ord(b) - 97] -= 1
        return all(c == 0 for c in count)

sol = Solution()
print(sol.isAnagram("anagram", "nagaram"), sol.isAnagram("rat", "car"))

True False


Bản ngắn gọn (cũng Accepted, và **đúng luôn với Unicode**):

```python
from collections import Counter

class Solution:
    def isAnagram(self, s: str, t: str) -> bool:
        return Counter(s) == Counter(t)
```

## 10. Vì sao đúng? Độ phức tạp?

**Invariant** (cách mảng): sau khi xử lý `i` cặp, `count[c]` = (số lần `c` trong `s[:i]`) − (số lần `c` trong `t[:i]`). Khi `i = n`, `count[c] = 0` với mọi `c` ⇔ hai chuỗi có cùng tần suất mọi ký tự ⇔ anagram.

Gọi `n = len(s)` (= `len(t)` nếu không thoát sớm), `k` = kích thước bảng chữ cái (26).

| Cách | Time | Space phụ |
|---|---|---|
| Sắp xếp | `O(n log n)` | `O(n)` (list mới) |
| Hai dict | `O(n)` | `O(k)` |
| Mảng 26, +1/-1 | `O(n + k)` = `O(n)` | `O(k)` = `O(1)` |
| `Counter` | `O(n)` trung bình (băm) | `O(k)` |
| Trừ dần + dừng sớm | `O(n)` | `O(k)` |

`O(1)` bộ nhớ ở cách mảng vì 26 là hằng số, không phụ thuộc `n`.

## 11. Follow-up: Unicode

Mảng 26 ô chỉ đúng với `a..z`. Với Unicode (chữ có dấu tiếng Việt, emoji, chữ Hán...):

- Dùng **hash map** (`dict`/`Counter`) — key là ký tự bất kỳ, không cần biết trước bảng chữ cái.
- Lưu ý chuẩn hóa: `"é"` có thể viết là 1 code point (`U+00E9`) hoặc 2 (`e` + dấu `U+0301`). Hai chuỗi "nhìn giống nhau" có thể khác ở mức code point → dùng `unicodedata.normalize("NFC", s)` trước khi đếm.

In [15]:
import unicodedata
a = "é"                    # 1 code point
b = "é"              # e + dấu sắc kết hợp
print(len(a), len(b), a == b)
print("Counter thô       :", Counter(a) == Counter(b))
na, nb = unicodedata.normalize("NFC", a), unicodedata.normalize("NFC", b)
print("Counter sau NFC   :", Counter(na) == Counter(nb))
print("Tiếng Việt        :", Counter("hồng") == Counter("ngồh"))
try:
    is_anagram_array("hồng", "ngồh")
except IndexError as e:
    print("Mảng 26 ô với Unicode -> IndexError:", e)

1 2 False
Counter thô       : False
Counter sau NFC   : True
Tiếng Việt        : True
Mảng 26 ô với Unicode -> IndexError: list index out of range


## 12. Bộ test

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}")

sol = Solution()
check("ví dụ 1", sol.isAnagram("anagram", "nagaram"), True)
check("ví dụ 2", sol.isAnagram("rat", "car"), False)
check("một ký tự giống", sol.isAnagram("a", "a"), True)
check("một ký tự khác", sol.isAnagram("a", "b"), False)
check("cùng bộ chữ, khác số lần", sol.isAnagram("aab", "abb"), False)
check("khác độ dài", sol.isAnagram("ab", "abc"), False)
check("chuỗi giống hệt", sol.isAnagram("listen", "listen"), True)
check("listen/silent", sol.isAnagram("listen", "silent"), True)
check("dài 5*10^4", sol.isAnagram("ab" * 25000, "ba" * 25000), True)

PASS  ví dụ 1
PASS  ví dụ 2
PASS  một ký tự giống
PASS  một ký tự khác
PASS  cùng bộ chữ, khác số lần
PASS  khác độ dài
PASS  chuỗi giống hệt
PASS  listen/silent
PASS  dài 5*10^4


In [17]:
import random, string
rng = random.Random(242)
solvers = [is_anagram_sort, is_anagram_two_dicts, is_anagram_array,
           is_anagram_counter, is_anagram_early_exit, Solution().isAnagram]
for _ in range(2000):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    r = rng.random()
    if r < 0.4:
        t = "".join(rng.sample(s, len(s)))             # hoán vị -> anagram
    elif r < 0.7:
        lst = list(s); lst[rng.randrange(len(lst))] = rng.choice("abc"); t = "".join(lst)
    else:
        t = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    outs = {f.__name__: f(s, t) for f in solvers}
    assert len(set(outs.values())) == 1, (s, t, outs)
print("2000 cặp ngẫu nhiên: 6 cài đặt cho cùng kết quả — PASS")

2000 cặp ngẫu nhiên: 6 cài đặt cho cùng kết quả — PASS


`rng.sample(s, len(s))` lấy ngẫu nhiên **không lặp** đủ `len(s)` ký tự = một hoán vị của `s` → chắc chắn là anagram. Seed cố định (`242`) để mỗi lần chạy cho cùng dữ liệu.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Chỉ so **tập** ký tự

In [18]:
def wrong_set(s, t):
    return set(s) == set(t)

print("aab/abb ->", wrong_set("aab", "abb"), "(phải False)")

aab/abb -> True (phải False)


**Vì sao sai:** `set` bỏ trùng, mất thông tin số lần. **Sửa:** đếm tần suất (`Counter`) hoặc sắp xếp.

### Lỗi 2: Chỉ kiểm tra "mỗi ký tự của `t` có trong `s`"

In [19]:
def wrong_membership(s, t):
    return all(ch in s for ch in t)

print("a/aaa ->", wrong_membership("a", "aaa"), "(phải False)")

a/aaa -> True (phải False)


**Sửa:** kiểm tra số lần, và độ dài.

### Lỗi 3: Dùng `count[ch] += 1` với dict thường mà chưa khởi tạo

In [20]:
d = {}
try:
    d["x"] += 1
except KeyError as e:
    print("KeyError:", e, "-> dùng d.get(ch, 0) + 1 hoặc Counter/defaultdict")

KeyError: 'x' -> dùng d.get(ch, 0) + 1 hoặc Counter/defaultdict


### Lỗi 4: Gọi `str.count` trong vòng lặp → `O(n²)` hoặc tệ hơn

```python
for ch in s:
    if s.count(ch) != t.count(ch):   # mỗi lần count duyệt cả chuỗi
        return False
```

Đúng nhưng chậm: `n` lần × `O(n)` mỗi lần = `O(n²)`; với `n = 5*10^4` là ~2.5 tỉ thao tác. Nếu muốn dùng `str.count`, chỉ lặp qua **26 chữ cái**: `all(s.count(c) == t.count(c) for c in string.ascii_lowercase)` → `O(26n)`.

In [21]:
print(all(s_.count(c) == t_.count(c) for s_, t_ in [("anagram", "nagaram")] for c in string.ascii_lowercase))

True


### Lỗi 5: Quên `len(s) != len(t)` khi dùng `zip`

`zip` dừng ở chuỗi **ngắn hơn**. Không kiểm tra độ dài thì `("ab", "abc")` sẽ chỉ xét `a,b` → mảng đếm toàn 0 → trả `True` sai.

In [22]:
def wrong_no_len(s, t):
    count = [0] * 26
    for a, b in zip(s, t):
        count[ord(a) - 97] += 1
        count[ord(b) - 97] -= 1
    return all(c == 0 for c in count)

print("ab/abc ->", wrong_no_len("ab", "abc"), "(phải False)")

ab/abc -> True (phải False)


## 14. Mẹo nhận dạng pattern

- "Hoán vị", "sắp xếp lại các chữ", "cùng ký tự" → **đếm tần suất** (frequency count / hash map).
- Bảng chữ cái nhỏ, cố định (26 chữ thường, 10 chữ số) → **mảng đếm** thay dict.
- "So hai bảng đếm" → một bảng `+1/-1` rồi kiểm tra toàn 0.
- Khóa chuẩn hóa cho nhóm anagram: `"".join(sorted(w))` hoặc `tuple(count)` — dùng ở bài **Group Anagrams (49)**.

## 15. Biến thể

In [23]:
from collections import defaultdict

def group_anagrams(words):
    groups = defaultdict(list)          # key chưa có -> tự tạo list rỗng
    for w in words:
        key = tuple(sorted(w))
        groups[key].append(w)
    return list(groups.values())

def find_anagram_starts(s, p):
    """LeetCode 438: vị trí bắt đầu các đoạn con của s là anagram của p (cửa sổ trượt)."""
    if len(p) > len(s):
        return []
    need = Counter(p)
    window = Counter(s[:len(p)])
    res = [0] if window == need else []
    for i in range(len(p), len(s)):
        window[s[i]] += 1
        window[s[i - len(p)]] -= 1
        if window[s[i - len(p)]] == 0:
            del window[s[i - len(p)]]
        if window == need:
            res.append(i - len(p) + 1)
    return res

print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))
print(find_anagram_starts("cbaebabacd", "abc"))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]
[0, 6]


`defaultdict(list)` giống dict nhưng truy cập key chưa có sẽ tự tạo `[]`. Trong cửa sổ trượt, ta `del` key về 0 để phép so sánh `window == need` không bị ảnh hưởng (thực ra `Counter` từ Python 3.10 coi key giá trị 0 như vắng mặt khi so sánh, nhưng xóa đi cho rõ ràng).

## 16. Bài tập tự luyện (TODO)

### Bài 1 — Đoán output
`sorted("cab")`, `Counter("cab") == Counter("abc")`, `set("aab") == set("ab")`, `ord("e") - ord("a")`.

### Bài 2 — Đếm tần suất bằng dict

In [24]:
def exercise_freq(text: str) -> dict:
    # TODO: dùng d.get(ch, 0) + 1
    pass

# assert exercise_freq("banana") == {"b": 1, "a": 3, "n": 2}

### Bài 3 — Tự viết lời giải mảng 26 ô

In [25]:
def exercise_is_anagram(s: str, t: str) -> bool:
    # TODO 1: so độ dài
    # TODO 2: count = [0] * 26; +1 cho s, -1 cho t
    # TODO 3: kiểm tra mọi ô bằng 0
    pass

# assert exercise_is_anagram("anagram", "nagaram") == True
# assert exercise_is_anagram("aab", "abb") == False

### Bài 4 — Anagram bỏ qua khoảng trắng và hoa/thường
`"Dormitory"` và `"Dirty room"` → `True`.

In [26]:
def exercise_is_anagram_phrase(a: str, b: str) -> bool:
    # TODO: lower(), bỏ ' ', rồi so Counter
    pass

# assert exercise_is_anagram_phrase("Dormitory", "Dirty room") == True
# assert exercise_is_anagram_phrase("Hello", "Olelh!") == False

### Bài 5 — Số ký tự tối thiểu cần đổi để `t` thành anagram của `s` (LeetCode 1347)
Hai chuỗi cùng độ dài. Mỗi bước thay một ký tự của `t` bằng ký tự bất kỳ. Ví dụ `s="bab", t="aba"` → `1`.

In [27]:
def exercise_min_steps(s: str, t: str) -> int:
    # TODO: đếm s - t; cộng các phần dương
    pass

# assert exercise_min_steps("bab", "aba") == 1
# assert exercise_min_steps("leetcode", "practice") == 5

### Bài 6 — Mini challenge: Group Anagrams không dùng `sorted`
Viết `group_anagrams_count(words)` với key là **tuple 26 số đếm** (`O(n·L)` thay vì `O(n·L log L)`).

In [28]:
def exercise_group_anagrams_count(words: list[str]) -> list[list[str]]:
    # TODO: key = tuple(count 26 ô)
    pass

# res = exercise_group_anagrams_count(["eat","tea","tan","ate","nat","bat"])
# assert sorted(map(sorted, res)) == [["ate","eat","tea"], ["bat"], ["nat","tan"]]

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Nhớ `sorted` trả list | `set` bỏ trùng | `['a','b','c']`, `True`, `True`, `4` |
| 2 | Mục 2.2 | vòng `for ch in text` | `d[ch] = d.get(ch, 0) + 1` |
| 3 | Mục 6 | `zip(s, t)` | `all(c == 0 for c in count)` |
| 4 | Chuẩn hóa trước | `x.lower().replace(" ", "")` | `Counter(...) == Counter(...)` |
| 5 | Ký tự `t` thiếu so với `s` phải được thay | `Counter(s) - Counter(t)` giữ phần dương | `sum((Counter(s) - Counter(t)).values())` |
| 6 | Hai từ anagram ⇔ cùng mảng đếm | list không làm key được, tuple thì được | `groups[tuple(cnt)].append(w)` |

## 18. Đáp án tham khảo, template và checklist

Bài 1: `['a', 'b', 'c']`, `True`, `True`, `4`.

In [29]:
def answer_freq(text):
    d = {}
    for ch in text:
        d[ch] = d.get(ch, 0) + 1
    return d

def answer_is_anagram(s, t):
    if len(s) != len(t):
        return False
    count = [0] * 26
    for a, b in zip(s, t):
        count[ord(a) - 97] += 1
        count[ord(b) - 97] -= 1
    return all(c == 0 for c in count)

def answer_is_anagram_phrase(a, b):
    norm = lambda x: x.lower().replace(" ", "")
    return Counter(norm(a)) == Counter(norm(b))

def answer_min_steps(s, t):
    return sum((Counter(s) - Counter(t)).values())

def answer_group_anagrams_count(words):
    groups = defaultdict(list)
    for w in words:
        cnt = [0] * 26
        for ch in w:
            cnt[ord(ch) - 97] += 1
        groups[tuple(cnt)].append(w)
    return list(groups.values())

print(sorted("cab"), Counter("cab") == Counter("abc"), set("aab") == set("ab"), ord("e") - ord("a"))
assert answer_freq("banana") == {"b": 1, "a": 3, "n": 2}
assert answer_is_anagram("anagram", "nagaram") and not answer_is_anagram("aab", "abb")
assert answer_is_anagram_phrase("Dormitory", "Dirty room") and not answer_is_anagram_phrase("Hello", "Olelh!")
assert answer_min_steps("bab", "aba") == 1 and answer_min_steps("leetcode", "practice") == 5
assert answer_min_steps("anagram", "mangaar") == 0
res = answer_group_anagrams_count(["eat", "tea", "tan", "ate", "nat", "bat"])
assert sorted(map(sorted, res)) == [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]
for _ in range(300):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 6)))
    t = "".join(rng.choice("ab") for _ in range(rng.randint(1, 6)))
    assert answer_is_anagram(s, t) == is_anagram_sort(s, t)
print("Đáp án tham khảo: tất cả test PASS")

['a', 'b', 'c'] True True 4
Đáp án tham khảo: tất cả test PASS


`Counter(s) - Counter(t)` là phép **trừ Counter**: trừ từng key và **chỉ giữ số dương** — chính là các ký tự `s` cần mà `t` thiếu.

### Template nhớ nhanh

```python
if len(s) != len(t):
    return False
count = [0] * 26
for a, b in zip(s, t):
    count[ord(a) - 97] += 1
    count[ord(b) - 97] -= 1
return all(c == 0 for c in count)
# Unicode: return Counter(s) == Counter(t)
```

### Checklist submit

- [ ] So độ dài trước (nhất là khi dùng `zip`).
- [ ] Đếm **số lần**, không chỉ tập ký tự.
- [ ] Mảng 26 ô chỉ cho chữ thường `a..z`; Unicode → `dict`/`Counter` (+ chuẩn hóa NFC nếu cần).
- [ ] Không gọi `str.count` trong vòng lặp qua từng ký tự của chuỗi dài.
- [ ] Nêu được Time `O(n)`, Space `O(1)` (26 ô) cho cách tối ưu.